In [ ]:
SELECT 
  SALES_ID
 ,CUSTOMER_ID
 ,PRODUCT_ID
 ,STORE_ID
 ,ORDER_DATE
 ,TRY_CAST(QUANTITY AS NUMBER) AS QUANTITY  
 ,UNIT_PRICE,DISCOUNT
 ,TOTAL_AMOUNT 
 FROM SALES_DB.RAW.SALES WHERE CUSTOMER_ID IS NULL LIMIT 100; 

## Quarantine Bad Records
Move records with invalid QUANTITY (NULL, non-numeric, or negative) and NULL CUSTOMER_ID into a quarantine table with a reason column.

In [ ]:
%%sql -r quarantine_create
CREATE TABLE IF NOT EXISTS SALES_DB.RAW.SALES_QUARANTINE AS
SELECT
  *,
  CASE
    WHEN CUSTOMER_ID IS NULL AND (QUANTITY IS NULL OR TRY_CAST(QUANTITY AS NUMBER) IS NULL OR TRY_CAST(QUANTITY AS NUMBER) <= 0)
      THEN 'NULL CUSTOMER_ID and INVALID QUANTITY'
    WHEN CUSTOMER_ID IS NULL
      THEN 'NULL CUSTOMER_ID'
    WHEN QUANTITY IS NULL OR TRY_CAST(QUANTITY AS NUMBER) IS NULL
      THEN 'NON-NUMERIC or NULL QUANTITY'
    WHEN TRY_CAST(QUANTITY AS NUMBER) <= 0
      THEN 'NEGATIVE or ZERO QUANTITY'
  END AS QUARANTINE_REASON,
  CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM SALES_DB.RAW.SALES
WHERE CUSTOMER_ID IS NULL
   OR QUANTITY IS NULL
   OR TRY_CAST(QUANTITY AS NUMBER) IS NULL
   OR TRY_CAST(QUANTITY AS NUMBER) <= 0;

In [ ]:
%%sql -r quarantine_summary
SELECT QUARANTINE_REASON, COUNT(*) AS BAD_RECORD_COUNT
FROM SALES_DB.RAW.SALES_QUARANTINE
GROUP BY QUARANTINE_REASON
ORDER BY BAD_RECORD_COUNT DESC;

In [ ]:
%%sql -r quarantine_preview
SELECT * FROM SALES_DB.RAW.SALES_QUARANTINE LIMIT 20;

## Store Clean Data in Silver Layer
Filter out all quarantined records and cast columns to proper types.

In [ ]:
%%sql -r silver_create
CREATE OR REPLACE TABLE SALES_DB.SILVER.SALES AS
SELECT
  SALES_ID,
  CUSTOMER_ID,
  PRODUCT_ID,
  STORE_ID,
  TRY_CAST(ORDER_DATE AS DATE) AS ORDER_DATE,
  TRY_CAST(QUANTITY AS NUMBER) AS QUANTITY,
  TRY_CAST(UNIT_PRICE AS FLOAT) AS UNIT_PRICE,
  TRY_CAST(DISCOUNT AS FLOAT) AS DISCOUNT,
  TRY_CAST(TOTAL_AMOUNT AS FLOAT) AS TOTAL_AMOUNT
FROM SALES_DB.RAW.SALES
WHERE CUSTOMER_ID IS NOT NULL
  AND QUANTITY IS NOT NULL
  AND TRY_CAST(QUANTITY AS NUMBER) IS NOT NULL
  AND TRY_CAST(QUANTITY AS NUMBER) > 0;

In [ ]:
%%sql -r row_counts
SELECT
  (SELECT COUNT(*) FROM SALES_DB.RAW.SALES) AS RAW_COUNT,
  (SELECT COUNT(*) FROM SALES_DB.RAW.SALES_QUARANTINE) AS QUARANTINED_COUNT,
  (SELECT COUNT(*) FROM SALES_DB.SILVER.SALES) AS SILVER_COUNT;